# Lesson 1: Router Engine

Welcome to Lesson 1.

To access the `requirements.txt` file, the data/pdf file required for this lesson and the `helper` and `utils` modules, please go to the `File` menu and select`Open...`.

I hope you enjoy this course!

## Setup

In [ ]:
# from helper import get_openai_api_key

# OPENAI_API_KEY = get_openai_api_key()

In [1]:
import nest_asyncio

nest_asyncio.apply()

## Load Data

Para baixar este artigo, abaixo está o código necessário:

#!wget "https://openreview.net/pdf/474fc6dad3bd9bf7fdb97c7cd72b2cc0649a9647.pdf" -O metagpt.pdf

**Nota**: O arquivo pdf está incluso nesta lição. Para acessá-lo, vá ao menu `Arquivo` e selecione `Abrir...`.

In [ ]:
# ! pip install -q llama-index

In [2]:
from llama_index.core import SimpleDirectoryReader

# load documents
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

## Define LLM and Embedding model

In [3]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [6]:
# nodes

In [4]:
import os

os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

In [9]:
# %pip install -q llama-index-llms-anthropic llama-index-embeddings-huggingface

In [5]:
from llama_index.core import Settings
from llama_index.llms.anthropic import Anthropic
# from llama_index.embeddings.openai import OpenAIEmbedding

# Settings.llm = OpenAI(model="gpt-3.5-turbo")
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

Settings.llm = Anthropic(model="claude-3-haiku-20240307")
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Defina o Índice de Resumo e o Índice de Vetor sobre os Mesmos Dados

In [6]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

## Definir mecanismos de consulta e definir metadados

In [7]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)

vector_query_engine = vector_index.as_query_engine()

In [8]:
from llama_index.core.tools import QueryEngineTool


summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Útil para questões de resumo relacionadas ao MetaGPT"
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Útil para recuperar contexto específico do artigo MetaGPT."
    ),
)

## Definir mecanismo de consulta do roteador

In [9]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True
)

In [10]:
response = query_engine.query("Qual é o resumo do documento?")
print(str(response))

Selecting query engine 0: The summary of the document is most relevant to the first choice, which states that it is 'Útil para questões de resumo relacionadas ao MetaGPT'..
De acordo com as informações fornecidas, o documento é um preprint sobre o MetaGPT, um framework de meta-programação para sistemas multi-agentes colaborativos baseados em modelos de linguagem de grande porte (LLMs). O MetaGPT incorpora procedimentos operacionais padronizados (SOPs) humanos em suas interações entre agentes, a fim de melhorar a robustez e reduzir inconsistências lógicas. Ele utiliza um paradigma de linha de montagem para atribuir diferentes papéis a vários agentes, decompondo eficientemente tarefas complexas em subtarefas. O MetaGPT alcança um novo estado da arte em benchmarks de geração de código, como HumanEval e MBPP, em comparação com outros sistemas multi-agentes baseados em LLM.


In [14]:
response.response

'De acordo com as informações fornecidas, o documento é um preprint sobre o MetaGPT, um framework de meta-programação para sistemas multi-agentes colaborativos baseados em modelos de linguagem de grande porte (LLMs). O MetaGPT incorpora procedimentos operacionais padronizados (SOPs) humanos em suas interações entre agentes, a fim de melhorar a robustez e reduzir inconsistências lógicas. Ele utiliza um paradigma de linha de montagem para atribuir diferentes papéis a vários agentes, decompondo eficientemente tarefas complexas em subtarefas. O MetaGPT alcança um novo estado da arte em benchmarks de geração de código, como HumanEval e MBPP, em comparação com outros sistemas multi-agentes baseados em LLM.'

In [11]:
print(len(response.source_nodes))

34


In [16]:
response = query_engine.query(
    "Como os agentes compartilham informações com outros agentes?"
)
print(response.response)

Selecting query engine 1: The given choices are related to the MetaGPT article, and the question 'Como os agentes compartilham informações com outros agentes?' is more relevant to the context of the MetaGPT article, which is about how agents share information with other agents..
Os agentes compartilham informações através de um mecanismo de publicação e assinatura. Eles publicam suas mensagens estruturadas em um pool de mensagens global compartilhado, permitindo que todos os agentes acessem as informações diretamente. Isso elimina a necessidade de solicitar informações a outros agentes e aguardar suas respostas, aumentando a eficiência da comunicação.

Além disso, os agentes utilizam um mecanismo de assinatura para receber apenas as informações relevantes para suas funções, evitando sobrecarga de informações. Eles podem selecionar quais informações desejam acompanhar com base em seus perfis de função. Dessa forma, cada agente recebe apenas os dados necessários para suas tarefas, melhor

## Vamos juntar tudo

In [20]:
def get_router_query_engine(file_path: str, llm = None, embed_model = None):
    """Get router query engine."""

    # load documents
    documents = SimpleDirectoryReader(input_files=[file_path]).load_data()

    splitter = SentenceSplitter(chunk_size=1024)
    nodes = splitter.get_nodes_from_documents(documents)

    summary_index = SummaryIndex(nodes)
    vector_index = VectorStoreIndex(nodes, embed_model=embed_model)

    summary_query_engine = summary_index.as_query_engine(
        response_mode="tree_summarize",
        use_async=True,
        llm=llm
    )
    vector_query_engine = vector_index.as_query_engine(llm=llm)

    summary_tool = QueryEngineTool.from_defaults(
        query_engine=summary_query_engine,
        description=(
            "Useful for summarization questions related to MetaGPT"
        ),
    )

    vector_tool = QueryEngineTool.from_defaults(
        query_engine=vector_query_engine,
        description=(
            "Useful for retrieving specific context from the MetaGPT paper."
        ),
    )

    query_engine = RouterQueryEngine(
        selector=LLMSingleSelector.from_defaults(),
        query_engine_tools=[
            summary_tool,
            vector_tool,
        ],
        verbose=True
    )
    return query_engine

In [19]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
llm = Anthropic(model="claude-3-haiku-20240307")
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

In [22]:
query_engine = get_router_query_engine("metagpt.pdf", llm, embed_model)

In [24]:
response = query_engine.query("Conte-me sobre os resultados do estudo de ablação?")
print(str(response))

Selecting query engine 1: The question is asking for specific details from the MetaGPT paper, which is better suited for the second choice that is 'Useful for retrieving specific context from the MetaGPT paper.'.
Infelizmente não há informações sobre um estudo de ablação no contexto fornecido. O texto se concentra em comparar a executabilidade de diferentes sistemas de inteligência artificial em tarefas de desenvolvimento de software, bem como discutir os desafios e soluções do MetaGPT. Não há menção a nenhum estudo de ablação.
